In [7]:
import sys

import numpy as np
import pandas as pd


import plotly.offline as py

py.init_notebook_mode(connected=True)
import plotly.graph_objs as go
import plotly.tools as tls
import seaborn as sns
import matplotlib.pyplot as plt


from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from imblearn.over_sampling import SMOTE
import xgboost as xgb
from sklearn.model_selection import train_test_split

# Import and suppress warnings
import warnings

warnings.filterwarnings("ignore")

In [5]:
df = pd.read_csv("../data/HR-Employee-Attrition.csv")

In [6]:
df.head(10)

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2
5,32,No,Travel_Frequently,1005,Research & Development,2,2,Life Sciences,1,8,...,3,80,0,8,2,2,7,7,3,6
6,59,No,Travel_Rarely,1324,Research & Development,3,3,Medical,1,10,...,1,80,3,12,3,2,1,0,0,0
7,30,No,Travel_Rarely,1358,Research & Development,24,1,Life Sciences,1,11,...,2,80,1,1,2,3,1,0,0,0
8,38,No,Travel_Frequently,216,Research & Development,23,3,Life Sciences,1,12,...,2,80,0,10,2,3,9,7,1,8
9,36,No,Travel_Rarely,1299,Research & Development,27,3,Medical,1,13,...,2,80,2,17,3,2,7,7,7,7


In [9]:
df.isna().sum().any()

np.False_

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   object
 2   BusinessTravel            1470 non-null   object
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   object
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   int64 
 7   EducationField            1470 non-null   object
 8   EmployeeCount             1470 non-null   int64 
 9   EmployeeNumber            1470 non-null   int64 
 10  EnvironmentSatisfaction   1470 non-null   int64 
 11  Gender                    1470 non-null   object
 12  HourlyRate                1470 non-null   int64 
 13  JobInvolvement            1470 non-null   int64 
 14  JobLevel                

In [11]:
categorical = df.select_dtypes(include=["object"])

In [12]:
categorical.head(10)

,Attrition,BusinessTravel,Department,EducationField,Gender,JobRole,MaritalStatus,Over18,OverTime
0,Yes,Travel_Rarely,Sales,Life Sciences,Female,Sales Executive,Single,Y,Yes
1,No,Travel_Frequently,Research & Development,Life Sciences,Male,Research Scientist,Married,Y,No
2,Yes,Travel_Rarely,Research & Development,Other,Male,Laboratory Technician,Single,Y,Yes
3,No,Travel_Frequently,Research & Development,Life Sciences,Female,Research Scientist,Married,Y,Yes
4,No,Travel_Rarely,Research & Development,Medical,Male,Laboratory Technician,Married,Y,No
5,No,Travel_Frequently,Research & Development,Life Sciences,Male,Laboratory Technician,Single,Y,No
6,No,Travel_Rarely,Research & Development,Medical,Female,Laboratory Technician,Married,Y,Yes
7,No,Travel_Rarely,Research & Development,Life Sciences,Male,Laboratory Technician,Divorced,Y,No
8,No,Travel_Frequently,Research & Development,Life Sciences,Male,Manufacturing Director,Single,Y,No
9,No,Travel_Rarely,Research & Development,Medical,Male,Healthcare Representative,Married,Y,No


In [39]:
categorical = pd.get_dummies(categorical, drop_first=True, dtype=int)

## GBM Classifier

#### 1.n_estimators - No of Trees in the Model

###### 2.max_features - The number of features to consider while searching for a best split.Thumb Rule to have Square root of no of Columns

##### 3.max_depth - Maximum Depth of Tree and can be used to control overfiting

##### 4.min_samples_leaf - Minimum samples (or observations) required in a terminal node or leaf.In general we need to have lower values  for it for Imbalanced problems

##### 5.subsample- The fraction of samples to be used for fitting the individual base learners

##### 6.learning_rate - Learning rate shrinks the contribution of each tree by learning_rate. There is a trade-off between learning_rate and n_estimators

In [14]:
help(GradientBoostingClassifier)

Help on class GradientBoostingClassifier in module sklearn.ensemble._gb:

class GradientBoostingClassifier(sklearn.base.ClassifierMixin, BaseGradientBoosting)
 |  GradientBoostingClassifier(*, loss='log_loss', learning_rate=0.1, n_estimators=100, subsample=1.0, criterion='friedman_mse', min_samples_split=2, min_samples_leaf=1, min_weight_fraction_leaf=0.0, max_depth=3, min_impurity_decrease=0.0, init=None, random_state=None, max_features=None, verbose=0, max_leaf_nodes=None, warm_start=False, validation_fraction=0.1, n_iter_no_change=None, tol=0.0001, ccp_alpha=0.0)
 |
 |  Gradient Boosting for classification.
 |
 |  This algorithm builds an additive model in a forward stage-wise fashion; it
 |  allows for the optimization of arbitrary differentiable loss functions. In
 |  each stage ``n_classes_`` regression trees are fit on the negative gradient
 |  of the loss function, e.g. binary or multiclass log loss. Binary
 |  classification is a special case where only a single regression tre

In [16]:
import importlib
from utilties.utils import unique_values_percentage

help(unique_values_percentage)

Help on function unique_values_percentage in module utilties.utils:

unique_values_percentage(df, columns)
    A function to calculate the summary of categorical columns of a dataframe
        @param: df -> DataFrame
        @param columns: list of categorical columns in the dataframe
        @returns: dictionary of summary of unique value counts and their percentages
    >>> x = pd.DataFrame({'temp':['hot','hot','cool']})



In [19]:
import sys
from utilties import utils

importlib.reload(utils)

<module 'utilties.utils' from '/Users/deven/Developer/ML_AI/utilties/utils.py'>

In [22]:
gb = GradientBoostingClassifier(random_state=0)

list(gb.get_params().items())

[('ccp_alpha', 0.0),
 ('criterion', 'friedman_mse'),
 ('init', None),
 ('learning_rate', 0.1),
 ('loss', 'log_loss'),
 ('max_depth', 3),
 ('max_features', None),
 ('max_leaf_nodes', None),
 ('min_impurity_decrease', 0.0),
 ('min_samples_leaf', 1),
 ('min_samples_split', 2),
 ('min_weight_fraction_leaf', 0.0),
 ('n_estimators', 100),
 ('n_iter_no_change', None),
 ('random_state', 0),
 ('subsample', 1.0),
 ('tol', 0.0001),
 ('validation_fraction', 0.1),
 ('verbose', 0),
 ('warm_start', False)]

In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   object
 2   BusinessTravel            1470 non-null   object
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   object
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   int64 
 7   EducationField            1470 non-null   object
 8   EmployeeCount             1470 non-null   int64 
 9   EmployeeNumber            1470 non-null   int64 
 10  EnvironmentSatisfaction   1470 non-null   int64 
 11  Gender                    1470 non-null   object
 12  HourlyRate                1470 non-null   int64 
 13  JobInvolvement            1470 non-null   int64 
 14  JobLevel                

In [41]:
df_num = df.select_dtypes(include=["int64"])

In [42]:
df_num.head(10)

,Age,DailyRate,DistanceFromHome,Education,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,HourlyRate,JobInvolvement,JobLevel,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,1102,1,2,1,1,2,94,3,2,...,1,80,0,8,0,1,6,4,0,5
1,49,279,8,1,1,2,3,61,2,2,...,4,80,1,10,3,3,10,7,1,7
2,37,1373,2,2,1,4,4,92,2,1,...,2,80,0,7,3,3,0,0,0,0
3,33,1392,3,4,1,5,4,56,3,1,...,3,80,0,8,3,3,8,7,3,0
4,27,591,2,1,1,7,1,40,3,1,...,4,80,1,6,3,3,2,2,2,2
5,32,1005,2,2,1,8,4,79,3,1,...,3,80,0,8,2,2,7,7,3,6
6,59,1324,3,3,1,10,3,81,4,1,...,1,80,3,12,3,2,1,0,0,0
7,30,1358,24,1,1,11,4,67,3,1,...,2,80,1,1,2,3,1,0,0,0
8,38,216,23,3,1,12,4,44,2,3,...,2,80,0,10,2,3,9,7,1,8
9,36,1299,27,3,1,13,3,94,3,2,...,2,80,2,17,3,2,7,7,7,7


In [28]:
categorical.drop("Attrition", axis=1, inplace=True)

In [29]:
from sklearn import preprocessing

le = preprocessing.LabelEncoder()
help(le)

Help on LabelEncoder in module sklearn.preprocessing._label object:

class LabelEncoder(sklearn.base.TransformerMixin, sklearn.base.BaseEstimator)
 |  Encode target labels with value between 0 and n_classes-1.
 |
 |  This transformer should be used to encode target values, *i.e.* `y`, and
 |  not the input `X`.
 |
 |  Read more in the :ref:`User Guide <preprocessing_targets>`.
 |
 |  .. versionadded:: 0.12
 |
 |  Attributes
 |  ----------
 |  classes_ : ndarray of shape (n_classes,)
 |      Holds the label for each class.
 |
 |  See Also
 |  --------
 |  OrdinalEncoder : Encode categorical features using an ordinal encoding
 |      scheme.
 |  OneHotEncoder : Encode categorical features as a one-hot numeric array.
 |
 |  Examples
 |  --------
 |  `LabelEncoder` can be used to normalize labels.
 |
 |  >>> from sklearn.preprocessing import LabelEncoder
 |  >>> le = LabelEncoder()
 |  >>> le.fit([1, 2, 2, 6])
 |  LabelEncoder()
 |  >>> le.classes_
 |  array([1, 2, 6])
 |  >>> le.transform

In [35]:
le.fit_transform(["deven", "paris"])

array([0, 1])

In [31]:
le.fit_transform(cat)

array([0, 1, 2, 3, 4, 5, 6, 7, 8])

In [38]:
le.transform(["paris"])

array([1])

In [44]:
y = df["Attrition"]
y = y.apply(lambda x: 1 if x == "Yes" else 0)

In [45]:
df_final = pd.concat([categorical, df_num], axis=1)
df_final.head(10)

,BusinessTravel_Travel_Frequently,BusinessTravel_Travel_Rarely,Department_Research & Development,Department_Sales,EducationField_Life Sciences,EducationField_Marketing,EducationField_Medical,EducationField_Other,EducationField_Technical Degree,Gender_Male,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,0,1,0,1,1,0,0,0,0,0,...,1,80,0,8,0,1,6,4,0,5
1,1,0,1,0,1,0,0,0,0,1,...,4,80,1,10,3,3,10,7,1,7
2,0,1,1,0,0,0,0,1,0,1,...,2,80,0,7,3,3,0,0,0,0
3,1,0,1,0,1,0,0,0,0,0,...,3,80,0,8,3,3,8,7,3,0
4,0,1,1,0,0,0,1,0,0,1,...,4,80,1,6,3,3,2,2,2,2
5,1,0,1,0,1,0,0,0,0,1,...,3,80,0,8,2,2,7,7,3,6
6,0,1,1,0,0,0,1,0,0,0,...,1,80,3,12,3,2,1,0,0,0
7,0,1,1,0,1,0,0,0,0,1,...,2,80,1,1,2,3,1,0,0,0
8,1,0,1,0,1,0,0,0,0,1,...,2,80,0,10,2,3,9,7,1,8
9,0,1,1,0,0,0,1,0,0,1,...,2,80,2,17,3,2,7,7,7,7


In [46]:
train, test, target_train, target_test = train_test_split(
    df_final, y, train_size=0.75, random_state=0
);

In [48]:
train.shape

(1102, 47)

In [49]:
test.shape

(368, 47)

In [50]:
target_train.shape

(1102,)

In [51]:
gb.fit(train, target_train)

GradientBoostingClassifier(random_state=0)

In [53]:
gb_pred = gb.predict(test)

In [54]:
gb.score(train, target_train)

0.9627949183303085

In [55]:
gb.score(test, target_test)

0.8858695652173914

In [56]:
gb_prob = gb.predict_proba(test)

In [57]:
gb_prob

array([[0.93565701, 0.06434299],
       [0.96798638, 0.03201362],
       [0.8747258 , 0.1252742 ],
       [0.95525024, 0.04474976],
       [0.10720351, 0.89279649],
       [0.67487367, 0.32512633],
       [0.59570909, 0.40429091],
       [0.91183264, 0.08816736],
       [0.96662071, 0.03337929],
       [0.90249265, 0.09750735],
       [0.93460885, 0.06539115],
       [0.90269076, 0.09730924],
       [0.9732961 , 0.0267039 ],
       [0.26854533, 0.73145467],
       [0.94012005, 0.05987995],
       [0.98792486, 0.01207514],
       [0.94597233, 0.05402767],
       [0.93071641, 0.06928359],
       [0.93537954, 0.06462046],
       [0.92440318, 0.07559682],
       [0.60995072, 0.39004928],
       [0.95078893, 0.04921107],
       [0.96838558, 0.03161442],
       [0.97980593, 0.02019407],
       [0.41611433, 0.58388567],
       [0.74410279, 0.25589721],
       [0.95968266, 0.04031734],
       [0.97096781, 0.02903219],
       [0.28238165, 0.71761835],
       [0.96050297, 0.03949703],
       [0.

In [58]:
accuracy_score(gb_pred, target_test)

0.8858695652173914

In [63]:
trace = go.Scatter(
    y=gb.feature_importances_,
    x=df_final.columns.values,
    mode="markers",
    marker=dict(
        sizemode="diameter",
        sizeref=1.3,
        size=12,
        color=gb.feature_importances_,
        colorscale="Portland",
        showscale=True,
    ),
    text=df_final.columns.values,
)
data = [trace]

layout = go.Layout(
    autosize=True,
    title="GBM Model Feature Importance",
    hovermode="closest",
    xaxis=dict(ticklen=5, showgrid=False, zeroline=False, showline=False),
    yaxis=dict(
        title="Feature Importance",
        showgrid=False,
        zeroline=False,
        ticklen=5,
        gridwidth=2,
    ),
    showlegend=False,
)
fig = go.Figure(data=data, layout=layout)
py.iplot(fig, filename="scatter")

np.int64(0)